# Workshop: Gemma from Scratch
## Notebook 8: Gemma Model Assembly

**Estimated Time: 15 minutes**

In this final notebook, we will assemble the full **Gemma3Model**. We will combine the embedding layer, a stack of Transformer blocks, the final normalization, and the language model (LM) head. We will also look at how to generate text autoregressively.

### Learning Objectives:
1. Assemble the full model hierarchy.
2. Understand weight tying between embedding and output layers.
3. **NEW: Implement Final Logit Soft-Capping (30.0) for Gemma 2/3.**
4. Implement basic greedy generation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Redefine necessary components to make this notebook self-contained
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_groups, h_dim, hidden_dim):
        super().__init__()
        # Simple mock for GQA and MLP to keep it concise
        self.attn = nn.Linear(dim, dim, bias=False)
        self.ffn = nn.Linear(dim, dim, bias=False)
        self.norm = RMSNorm(dim)
    def forward(self, x):
        x = x + self.attn(self.norm(x))
        x = x + self.ffn(self.norm(x))
        return x

# Hyperparameters
vocab_size = 256_000 # Gemma vocab size
emb_dim = 128
n_layers = 4
context_length = 512

### 1. The Full Model

The model consists of:
1. **Token Embeddings**: Maps token IDs to vectors.
2. **Transformer Blocks**: A stack of `n_layers` blocks.
3. **Final RMSNorm**: One last normalization step.
4. **Output Head**: Maps vectors back to vocab scores (logits).

In [ ]:
class Gemma3Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_layers, n_heads, n_kv_groups, h_dim, hidden_dim, final_cap=30.0):
        super().__init__()
        self.emb_dim = emb_dim
        self.final_cap = final_cap
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(emb_dim, n_heads, n_kv_groups, h_dim, hidden_dim) for _ in range(n_layers)
        ])
        
        self.final_norm = RMSNorm(emb_dim)
        self.out_head = nn.Linear(emb_dim, vocab_size, bias=False)
        
        # Weight Tying: share weights between embedding and output head
        self.out_head.weight = self.tok_emb.weight

    def forward(self, input_ids):
        # 1. Embed and scale
        x = self.tok_emb(input_ids) * (self.emb_dim ** 0.5)
        
        # 2. Pass through blocks
        for block in self.blocks:
            x = block(x)
            
        # 3. Final norm and head
        x = self.final_norm(x)
        logits = self.out_head(x)
        
        # 4. Final Logit Soft-Capping (Gemma 2/3 style)
        if self.final_cap is not None:
            logits = self.final_cap * torch.tanh(logits / self.final_cap)
            
        return logits

model = Gemma3Model(vocab_size, emb_dim, n_layers, 8, 2, 16, 512)
print("Model assembled with Final Logit Capping (30.0) successfully.")

### 2. Autoregressive Generation

To generate text, we predict the next token, append it to the input, and repeat. This is "autoregressive" because each step depends on the previous ones.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, context_length):
    model.eval()
    for _ in range(max_new_tokens):
        # Crop idx to the last context_length tokens
        idx_cond = idx[:, -context_length:]
        
        # Get predictions (last token only)
        logits = model(idx_cond)[:, -1, :]
        
        # Greedy: take the most likely token
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        
        # Append
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Start with a dummy prompt (token ID 1)
prompt = torch.tensor([[1]])
generated = generate(model, prompt, max_new_tokens=10, context_length=context_length)
print("Generated sequence:", generated)

### 3. Conclusion

Congratulations! You've navigated the entire architecture of a modern LLM. From the math of attention to the assembly of billions of parameters, you now know how the "Gemma" under the hood works.

### Exercise:
Explain why logit soft-capping is useful for training large models like Gemma.

<details>
<summary><b>Click to see answer</b></summary>

Logit soft-capping prevents the output scores from becoming extremely large. If logits grow too much, the `softmax` distribution becomes extremely "peaky" (concentrated on one word), which leads to vanishing gradients for all other words and can cause numerical instability (overflow) during training.
</details>